# Musanga V5 — NB07 unified RGB-only production inference

Train one new all-reference model using NB03 architecture and label rules. No MS/DSM or historical generated data. Preserve NB04 held-out metrics separately. Native RGB grid, 100 m ROI interior, 256 px core / 64 px halo, probability and binary bands, -9999 NoData. Grouped reads and stripe-level resume; no per-tile Drive files. GPU required. Never run two writers concurrently.

## 00 — Native V5 source registry

In [ ]:
%pip -q install geopandas pyogrio shapely pandas numpy matplotlib rasterio segmentation-models-pytorch
from pathlib import Path
import json,math,hashlib
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),
                        Path('/content/drive/MyDrive/Musanga/Final/V5')] if p.exists()),None)
assert ROOT is not None
DATA=ROOT.parent/'Data'
OLD=ROOT/'Outputs'
AUDIT=ROOT/'Outputs/00_input_audit'
SPLIT_ROOT=ROOT/'Outputs/01_spatial_cover_folds_v2'
OUT=ROOT/'Outputs/07_RGB_unified_production_v5'
OUT.mkdir(parents=True,exist_ok=True)
V31=json.loads((OLD/'00_input_audit/00_input_manifest.json').read_text())
F78=json.loads((AUDIT/'00_v4_field78_contract.json').read_text())
SPLIT_CONTRACT=json.loads((SPLIT_ROOT/'01_reference_fold_contract_v2.json').read_text())
FOLD_CSV=Path(SPLIT_CONTRACT['output_csv'])
assert FOLD_CSV.exists(),FOLD_CSV
assert hashlib.sha256(FOLD_CSV.read_bytes()).hexdigest()==SPLIT_CONTRACT['csv_sha256'], 'NB01 fold file changed'
SPLITS=pd.read_csv(FOLD_CSV)
assert len(SPLITS)==108 and set(SPLITS.unit_type)=={'circle','plot'}
assert (SPLITS.unit_type=='circle').sum()==30 and (SPLITS.unit_type=='plot').sum()==78
assert SPLITS.groupby('spatial_group').fold.nunique().eq(1).all()
assert set(SPLITS.fold)=={0,1,2}
GEOMS=gpd.read_file(SPLIT_ROOT/'01_reference_groups_and_folds_v2.gpkg').to_crs('EPSG:32635')
assert len(GEOMS)==108
assert not GEOMS.duplicated(['unit_type','unit_id']).any()
SITES=['Baego','MOS','Yangambi','Yoko']
assert set(GEOMS.site)==set(SITES)
PLOT_PATH=Path(F78['plot_file'])
CROWN_PATH=Path(F78['exhaustive_plot_crowns'])
EXH_PATH=Path(F78['circle_positive_crowns'])
CIRCLE_PATH=Path(F78['circle_complete_roi'])
ROI_PATH=Path(V31['roi'])
for p in [PLOT_PATH,CROWN_PATH,EXH_PATH,CIRCLE_PATH,ROI_PATH]:assert p.exists(),p
print('REFERENCE:',len(GEOMS),'units;',GEOMS.spatial_group.nunique(),'spatial groups;',
      GEOMS.groupby('fold').size().to_dict(),'units by outer fold')
print('SPLIT FINGERPRINT:',SPLIT_CONTRACT['csv_sha256'][:20])
print('V3.1 KVO polygons:',EXH_PATH.name,'| V4 Nadine crowns:',CROWN_PATH.name)
print('OUT (new only):',OUT)
assert torch.cuda.is_available() if 'torch' in globals() else True
TRAIN=json.loads((ROOT/'Outputs/03_unified_rgb_unet_v5/03_training_contract.json').read_text())
assert TRAIN['model']=='unified' and TRAIN['frozen_nb01_hash']==SPLIT_CONTRACT['csv_sha256']
assert (ROOT/'Outputs/05_rgb_dsm_ms_matched_v1/06_reporting_contract.json').is_file()
import torch
assert torch.cuda.is_available(),'GPU runtime required'


## 01 — Reproduce annotation rules and full-data candidate selection

In [ ]:
# NB02.01 — Topology-safe vector audit; fail closed on material geometry changes.
from shapely.validation import explain_validity
from shapely import make_valid,union_all,set_precision
from shapely.geometry import GeometryCollection
from shapely.ops import unary_union
REF_CRS='EPSG:32635'
circles_full=gpd.read_file(CIRCLE_PATH).to_crs(REF_CRS)
circles_full['unit_id']=circles_full.index.astype(str)
circles=circles_full[circles_full.unit_id.isin(
    GEOMS.loc[GEOMS.unit_type.eq('circle'),'unit_id'])].copy()
plots=gpd.read_file(PLOT_PATH).to_crs(REF_CRS)
plots['unit_id']=plots.Plot_name.astype(str)
assert len(circles)==30 and len(plots)==78
assert plots.unit_id.is_unique and circles.unit_id.is_unique

def polygon_parts(geom):
    if geom is None or geom.is_empty:return []
    if geom.geom_type in ('Polygon','MultiPolygon'):return [geom]
    return [part for child in getattr(geom,'geoms',[]) for part in polygon_parts(child)]

QA02=[]
def repair_polygon_geoms(geoms,source,maximum_relative_change=.05):
    fixed=[]
    for ix,geom in enumerate(geoms):
        if geom is None or geom.is_empty:
            QA02.append(dict(source=source,row=ix,status='EMPTY_STOP',relative_area_change=np.nan,
                             original_valid=False,reason='empty'))
            raise ValueError(f'{source} polygon {ix} is empty; cannot silently drop reference')
        original_valid=bool(geom.is_valid)
        reason='valid' if original_valid else explain_validity(geom)
        candidate=geom if original_valid else make_valid(geom)
        parts=polygon_parts(candidate)
        if not parts:
            QA02.append(dict(source=source,row=ix,status='NO_POLYGON_STOP',relative_area_change=np.nan,
                             original_valid=original_valid,reason=reason))
            raise ValueError(f'{source} {ix} cannot be repaired into polygons: {reason}')
        try:
            candidate=unary_union(parts) if len(parts)>1 else parts[0]
        except Exception as exc:
            raise ValueError(f'Failed polygon repair for {source}, row {ix}: {exc}') from exc
        if not candidate.is_valid or candidate.is_empty:
            raise ValueError(f'Invalid repaired geometry: {source} row {ix}')
        # Invalid input polygon area is GEOS-defined but may be ambiguous; audit both.
        original_area=float(geom.area)
        new_area=float(candidate.area)
        change=abs(new_area-original_area)/max(abs(new_area),abs(original_area),1.e-9)
        status='unchanged' if original_valid else 'repaired'
        QA02.append(dict(source=source,row=ix,status=status,
            relative_area_change=change,original_valid=original_valid,reason=reason))
        if change>maximum_relative_change:
            raise ValueError(
                f'{source} row {ix}: repair changes polygon area by {change:.2%} '
                f'(limit {maximum_relative_change:.0%}). Review original shape manually.')
        fixed.append(candidate)
    return fixed

def robust_union02(geoms,source):
    geoms=[q for q in geoms if q is not None and not q.is_empty]
    if not geoms:return GeometryCollection()
    try:
        out=unary_union(geoms)
    except Exception as exc:
        print(f'WARNING {source}: exact union failed ({type(exc).__name__}); '
              'retrying 1-cm precision grid after individual geometry validation.')
        try:
            out=union_all(geoms,grid_size=.01)
        except Exception as exc2:
            raise RuntimeError(f'{source}: union failed at exact and 1-cm precision. '
                               'Review topological conflicts before training.') from exc2
    if not out.is_valid:
        raise ValueError(f'{source}: union is invalid; do not use for labels')
    return out

# Check source polygons BEFORE attempting any union. This is the fix for the
# TopologyException in the raw KVO exhaustive-positive shapefile.
exh=gpd.read_file(EXH_PATH).to_crs(REF_CRS)
exh.geometry=repair_polygon_geoms(exh.geometry,'V3_1_exhaustive_Musanga')
crowns=gpd.read_file(CROWN_PATH).to_crs(REF_CRS)
crowns['is_musanga']=pd.to_numeric(crowns.Class_ID,errors='raise').eq(1)
crowns.geometry=repair_polygon_geoms(crowns.geometry,'V4_Nadine_plot_crowns')
circles.geometry=repair_polygon_geoms(circles.geometry,'V3_1_circle_boundaries')
plots.geometry=repair_polygon_geoms(plots.geometry,'V4_plot_boundaries')
print('EXHAUSTIVE REFERENCE REPAIR SUMMARY (no rows dropped):')
display(pd.DataFrame(QA02).groupby(['source','status']).agg(
    n=('row','size'),max_rel_area_change=('relative_area_change','max')).round(5))
pd.DataFrame(QA02).to_csv(OUT/'02_source_geometry_repairs.csv',index=False)
print('Saved per-polygon repair audit:',OUT/'02_source_geometry_repairs.csv')

# Partial crown data can be quarantined under a separate, documented policy:
# a partial annotation is OPTIONAL supplementary supervision; a retained exhaustive
# circle or plot is NOT optional and must continue to fail closed on material repairs.
# Keep source-row indices intact so exclusions are traceable.
partial={}
partial_quarantine=[]
for site,path in V31['shared_annotations'].items():
    v=gpd.read_file(path).to_crs(REF_CRS)
    assert 'Class_ID' in v.columns
    v['musanga']=pd.to_numeric(v.Class_ID,errors='coerce').eq(1)
    retained=[]
    for source_row,geom in zip(v.index.tolist(),v.geometry.tolist()):
        label=f'V3_1_partial_{site}'
        original_valid=(geom is not None and not geom.is_empty and bool(geom.is_valid))
        reason=('valid' if original_valid else
                'empty' if geom is None or geom.is_empty else explain_validity(geom))
        change=np.nan
        try:
            if geom is None or geom.is_empty:
                raise ValueError('empty crown')
            candidate=geom if original_valid else make_valid(geom)
            parts=polygon_parts(candidate)
            if not parts:
                raise ValueError('no polygonal component after make_valid')
            candidate=unary_union(parts) if len(parts)>1 else parts[0]
            if candidate.is_empty or not candidate.is_valid:
                raise ValueError('invalid candidate after repair')
            change=abs(float(candidate.area)-float(geom.area))/max(
                float(candidate.area),float(geom.area),1e-9)
            if change>.05:
                raise ValueError(f'area change {change:.2%} exceeds 5%')
            retained.append((int(source_row),candidate))
            QA02.append(dict(source=label,row=int(source_row),
                status='unchanged' if original_valid else 'repaired',
                relative_area_change=change,original_valid=original_valid,reason=reason))
        except Exception as exc:
            QA02.append(dict(source=label,row=int(source_row),
                status='quarantined_partial',relative_area_change=change,
                original_valid=original_valid,reason=f'{reason}; {exc}'))
            partial_quarantine.append(dict(site=site,source_row=int(source_row),
                relative_area_change=change,original_valid=original_valid,
                reason=str(exc),source_path=str(path)))
    keep_ids=[idx for idx,_ in retained]
    v=v.loc[keep_ids].copy()
    v.geometry=[candidate for _,candidate in retained]
    assert v.geometry.is_valid.all()
    partial[site]=v
    print(f'PARTIAL {site}: retained {len(v)} / '
          f'{len(v)+sum(x["site"]==site for x in partial_quarantine)}; '
          f'quarantined {sum(x["site"]==site for x in partial_quarantine)}')
assert set(partial)==set(SITES)
assert all(len(v)>0 for v in partial.values()),'Missing usable partial crowns'
# Guard against a widespread underlying geometry/reference issue.
total_partial=sum(len(v) for v in partial.values())+len(partial_quarantine)
quarantine_rate=len(partial_quarantine)/total_partial
pd.DataFrame(QA02).to_csv(OUT/'02_source_geometry_repairs.csv',index=False)
pd.DataFrame(partial_quarantine,columns=[
    'site','source_row','relative_area_change','original_valid','reason','source_path'
]).to_csv(OUT/'02_partial_quarantine.csv',index=False)
print('PARTIAL QUARANTINE:',len(partial_quarantine),'/',total_partial,
      f'({quarantine_rate:.3%})')
display(pd.DataFrame(QA02).groupby(['source','status']).agg(
    n=('row','size'),max_rel_area_change=('relative_area_change','max')).round(5))
if quarantine_rate>.01:
    raise RuntimeError('More than 1% of partial crown polygons required quarantine. '
                       'Inspect 02_partial_quarantine.csv before proceeding.')

circle_union=robust_union02(list(circles.geometry),'circle footprints')
plot_union=robust_union02(list(plots.geometry),'plot footprints')
exh_union=robust_union02(list(exh.geometry),'KVO exhaustive-positive polygons')
circ_positive=exh_union.intersection(circle_union)
plot_pos=robust_union02(list(crowns.loc[crowns.is_musanga].geometry),
                        'Nadine positive crowns').intersection(plot_union)
plot_nonpos=robust_union02(list(crowns.loc[~crowns.is_musanga].geometry),
                           'Nadine negative crowns').intersection(plot_union)
shared=circle_union.intersection(plot_union)
# Compare definitions ONLY over their shared complete-annotation footprints.
circle_pos_shared=circ_positive.intersection(shared)
plot_pos_shared=plot_pos.intersection(shared)
diff_area=float(circle_pos_shared.symmetric_difference(plot_pos_shared).area)
shared_area=float(shared.area)
plot_internal_overlap=float(plot_pos.intersection(plot_nonpos).area)
print('Circle/plot complete-annotation overlap (m²):',round(shared_area,1))
print('Musanga label disagreement on overlap (m²):',round(diff_area,1))
print('Musanga disagreement % of shared area:',
      round(100*diff_area/shared_area,3) if shared_area else 0.)
print('Plot positive vs explicit negative crown overlap (m²):',
      round(plot_internal_overlap,2))
conflicts=circle_pos_shared.symmetric_difference(plot_pos_shared)
conflicts=conflicts.union(plot_pos.intersection(plot_nonpos))
assert conflicts.is_valid
reference_summary=dict(
  schema='musanga_v4_nb02_annotation_overlap_audit_v2',
  n_circles=30,n_plots=78,n_nadine_crowns=len(crowns),
  n_exhaustive_KVO_polygons=len(exh),
  source_geometry_repair_audit=str(OUT/'02_source_geometry_repairs.csv'),
  partial_quarantine_audit=str(OUT/'02_partial_quarantine.csv'),
  n_partial_quarantined=len(partial_quarantine),
  partial_quarantine_fraction=quarantine_rate,
  circle_plot_shared_area_m2=shared_area,
  exhaustive_positive_conflict_area_m2=diff_area,
  within_plot_positive_negative_overlap_m2=plot_internal_overlap,
  circle_authoritative='V3_1_KVO_final',
  plot_authoritative='V4_Nadine_exhaustive',
  conflict_policy='suppress contradicted supervised pixels (NOT relabel)',
  plot_background_negative_assumption='PENDING MANUAL SIGNOFF: annotated=True implies exhaustive canopy Musanga annotation')
(OUT/'02_reference_annotation_overlap_audit_v2.json').write_text(json.dumps(reference_summary,indent=2))
display(pd.DataFrame([reference_summary]).T.rename(columns={0:'value'}))
print('Review geometry repairs and overlap metrics before GPU training.')


In [ ]:
# NB02.02 — Distinguish RGB mosaic coverage from ROI exclusion and missing data.
RGB_DIR=DATA/'Ortho_RGB'
mosaics={}
for d in V31['mosaics']:
    name=d['file']
    if not name.endswith('.tif'):continue
    p=Path(d['path'])
    if not p.exists():p=RGB_DIR/name
    if not p.exists():raise FileNotFoundError(f'Missing RGB mosaic listed in V3.1: {p}')
    with rasterio.open(p) as ds:
        assert ds.crs is not None and ds.count>=3
        bounds=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(REF_CRS).iloc[0]
        mosaics[name]=dict(path=p,site=str(d['site']),bounds=bounds,
                           pixel_size_m=max(abs(ds.transform.a),abs(ds.transform.e)))
assert len(mosaics)==6,[(k,v['site']) for k,v in mosaics.items()]
assert set(v['site'] for v in mosaics.values())==set(SITES)
roi=gpd.read_file(ROI_PATH).to_crs(REF_CRS)
roi_union=robust_union02(list(roi.geometry),'general RGB ROI')
units=GEOMS.copy()
owners=[];all_options=[]
for u in units.itertuples(index=False):
    options=[]
    for name,d in mosaics.items():
        if d['site']!=u.site:continue
        footprint_ha=float(u.geometry.intersection(d['bounds']).area)/1e4
        roi_area_ha=float(u.geometry.intersection(d['bounds']).intersection(roi_union).area)/1e4
        area=max(float(u.geometry.area)/1e4,1.e-12)
        options.append(dict(unit_type=str(u.unit_type),unit_id=str(u.unit_id),
            site=str(u.site),fold=int(u.fold),mosaic=name,
            raw_RGB_bbox_coverage=footprint_ha/area,
            RGB_and_ROI_bbox_coverage=roi_area_ha/area))
    assert options
    all_options.extend(options)
    # Prefer the largest RGB+ROI geometric overlap; never accept zero-covered
    # assignments as valid merely because they are deterministic.
    best=sorted(options,key=lambda q:(q['RGB_and_ROI_bbox_coverage'],
                                     q['raw_RGB_bbox_coverage'],q['mosaic']),reverse=True)[0]
    owners.append(dict(unit_type=str(u.unit_type),unit_id=str(u.unit_id),
       site=str(u.site),fold=int(u.fold),spatial_group=str(u.spatial_group),
       mosaic=best['mosaic'],
       rgb_bbox_coverage_fraction=best['raw_RGB_bbox_coverage'],
       roi_mosaic_coverage_fraction=best['RGB_and_ROI_bbox_coverage']))
OWNERS=pd.DataFrame(owners)
OPTIONS=pd.DataFrame(all_options)
assert not OWNERS.duplicated(['unit_type','unit_id']).any()
print('RGB mosaics assigned to each site/reference type:')
display(OWNERS.groupby(['site','unit_type','mosaic']).agg(n=('unit_id','size'),
   min_roi_coverage=('roi_mosaic_coverage_fraction','min')).round(4))
PROBLEMS=OWNERS[OWNERS.roi_mosaic_coverage_fraction.lt(.95)].copy()
print('References with <95% geometric RGB + ROI coverage:',len(PROBLEMS))
display(PROBLEMS)
if len(PROBLEMS):
    print('All candidate mosaics for coverage-problem units:')
    display(OPTIONS.merge(PROBLEMS[['unit_type','unit_id']],on=['unit_type','unit_id'],
        validate='many_to_one').sort_values(['unit_id','RGB_and_ROI_bbox_coverage'],
        ascending=[True,False]).round(5))
    print('Interpretation: nonzero raw_RGB_bbox_coverage with zero RGB_and_ROI_bbox_coverage '
          'indicates ROI exclusion, not necessarily missing RGB data.')
OWNERS.to_csv(OUT/'02_reference_mosaic_owners_v2.csv',index=False)
OPTIONS.to_csv(OUT/'02_all_reference_mosaic_coverage_options.csv',index=False)
PROBLEMS.to_csv(OUT/'02_unresolved_rgb_coverage_units.csv',index=False)
# Only zero-coverage or low-coverage reference units are marked unavailable;
# preserve their NB01 fold assignments for provenance and spatial buffering.
UNAVAILABLE_KEYS=set(zip(PROBLEMS.unit_type.astype(str),PROBLEMS.unit_id.astype(str)))
assert set(zip(PROBLEMS.unit_type.astype(str),PROBLEMS.unit_id.astype(str)))=={('plot','Mixt1'),('plot','Mixt2')},(
    'Unexpected reference availability; investigate before freezing exclusions',PROBLEMS[['unit_type','unit_id']])
EXCLUSIONS=PROBLEMS[['unit_type','unit_id','site','fold','spatial_group','mosaic',
                     'rgb_bbox_coverage_fraction','roi_mosaic_coverage_fraction']].copy()
EXCLUSIONS['reason']='zero_RGB_bbox_intersection_in_all_site_mosaics'
EXCLUSIONS['eligible_training']=False
EXCLUSIONS['eligible_evaluation']=False
EXCLUSIONS.to_csv(OUT/'02_explicit_reference_exclusions.csv',index=False)
print('Fixed exclusion registry (fold geometry remains held out):')
display(EXCLUSIONS)
# Production sentinel fold 3 means all eligible references; no outer holdout.
HELDOUT={3:{site:GeometryCollection() for site in SITES}}
assert all(d['bounds'].is_valid for d in mosaics.values())


In [ ]:
CHIP=256
MAX_CANDIDATES_PER_UNIT=40
MAX_PARTIAL_PER_SITE=240
SEED=20261118
from rasterio.windows import Window
def candidate_chip_centres(ds,geom,max_chips):
    # Build a bounded grid of chip centers inside source geometry. The chip itself
    # may protrude; full-window holdout exclusion is checked separately.
    in_ds=gpd.GeoSeries([geom],crs=REF_CRS).to_crs(ds.crs).iloc[0]
    raw=rasterio.windows.from_bounds(*in_ds.bounds,transform=ds.transform)
    step=CHIP
    rmin=max(CHIP//2,int(np.floor(raw.row_off)))
    cmin=max(CHIP//2,int(np.floor(raw.col_off)))
    rmax=min(ds.height-CHIP//2,int(np.ceil(raw.row_off+raw.height)))
    cmax=min(ds.width-CHIP//2,int(np.ceil(raw.col_off+raw.width)))
    if rmax<rmin or cmax<cmin:return []
    rrange=list(range(rmin,rmax+1,step)) or [rmin]
    crange=list(range(cmin,cmax+1,step)) or [cmin]
    pts=[]
    for rr in rrange:
        for cc in crange:
            xy=ds.xy(rr,cc)
            point=gpd.GeoSeries(gpd.points_from_xy([xy[0]],[xy[1]]),crs=ds.crs).to_crs(REF_CRS).iloc[0]
            if geom.intersects(point):pts.append((rr,cc))
    # Always offer centroid so narrow or sub-chip-sized units yield a candidate.
    xy=in_ds.representative_point()
    row,col=ds.index(xy.x,xy.y)
    if CHIP//2<=row<ds.height-CHIP//2 and CHIP//2<=col<ds.width-CHIP//2:
        pts.append((row,col))
    pts=sorted(set(pts))
    if len(pts)>max_chips:
        take=np.linspace(0,len(pts)-1,max_chips).astype(int)
        pts=[pts[i] for i in take]
    return pts
records=[]
for fold in [3]:
    fold_units=units[units.fold.ne(fold)].copy()
    for mosaic,d in mosaics.items():
        site=d['site']
        heldout=HELDOUT[fold][site]
        assigned=OWNERS[(OWNERS.mosaic==mosaic)&OWNERS.fold.ne(fold)&
                        OWNERS.roi_mosaic_coverage_fraction.ge(.95)]
        with rasterio.open(d['path']) as ds:
            for unit in assigned.itertuples(index=False):
                geom=fold_units[(fold_units.unit_type==unit.unit_type)&
                    (fold_units.unit_id==unit.unit_id)].geometry.iloc[0]
                for rr,cc in candidate_chip_centres(ds,geom,MAX_CANDIDATES_PER_UNIT):
                    win=Window(int(cc-CHIP//2),int(rr-CHIP//2),CHIP,CHIP)
                    bbox=box(*rasterio.windows.bounds(win,ds.transform))
                    bb=gpd.GeoSeries([bbox],crs=ds.crs).to_crs(REF_CRS).iloc[0]
                    records.append(dict(fold=fold,site=site,mosaic=mosaic,
                       row=int(win.row_off),col=int(win.col_off),
                       source='circle' if unit.unit_type=='circle' else 'plot',
                       source_id=str(unit.unit_id),spatial_group=str(unit.spatial_group),
                       blocked_by_outer_holdout=bool(bb.intersects(heldout)),
                       full_chip_area_m2=float(bb.area)))
            # Sample explicit partial crowns only outside holdout buffers.
            crown=partial[site]
            if len(crown):
                z=crown[crown.geometry.intersects(d['bounds'])].copy()
                # Stable spatial candidate selection, no held-out labels consulted.
                if len(z)>MAX_PARTIAL_PER_SITE:
                    z=z.sample(n=MAX_PARTIAL_PER_SITE,random_state=SEED+fold)
                for ix,part in z.iterrows():
                    xy=part.geometry.representative_point()
                    r,c=ds.index(xy.x,xy.y)
                    win=Window(int(c-CHIP//2),int(r-CHIP//2),CHIP,CHIP)
                    if (win.row_off<0 or win.col_off<0 or
                        win.col_off+CHIP>ds.width or win.row_off+CHIP>ds.height):continue
                    bb=gpd.GeoSeries([box(*rasterio.windows.bounds(win,ds.transform))],
                                     crs=ds.crs).to_crs(REF_CRS).iloc[0]
                    records.append(dict(fold=fold,site=site,mosaic=mosaic,
                       row=int(win.row_off),col=int(win.col_off),source='partial',
                       source_id=str(ix),spatial_group='partial',
                       blocked_by_outer_holdout=bool(bb.intersects(heldout)),
                       full_chip_area_m2=float(bb.area)))
CANDIDATES=pd.DataFrame(records)
assert not CANDIDATES.empty
# A mosaic/window may overlap several original annotations: keep the first in
# each source class for source balancing; labels will be constructed from geometry.
CANDIDATES=CANDIDATES.drop_duplicates(['fold','site','mosaic','row','col','source']).copy()
CANDIDATES['eligible']=~CANDIDATES.blocked_by_outer_holdout
print('Candidate windows, by fold/source and leakage status:')
display(CANDIDATES.groupby(['fold','source','eligible']).size().unstack(fill_value=0))
print('Candidate mosaic attribution:')
display(CANDIDATES.loc[CANDIDATES.eligible].groupby(['fold','site','source']).size().unstack(fill_value=0))
if CANDIDATES.loc[CANDIDATES.eligible].empty:
    raise RuntimeError('No training candidates after holdout exclusion')
assert not CANDIDATES.loc[CANDIDATES.eligible].blocked_by_outer_holdout.any()
assert not any((str(r.source),str(r.source_id)) in UNAVAILABLE_KEYS for r in
    CANDIDATES.loc[CANDIDATES.eligible].itertuples(index=False)), 'Uncovered plot was sampled'
CANDIDATES.to_csv(OUT/'02_full_chip_heldout_exclusion_manifest.csv',index=False)
print('Manifest:',OUT/'02_full_chip_heldout_exclusion_manifest.csv')
print('Important: this is a candidate manifest, NOT a guarantee of supervised pixel counts.')


In [ ]:
from rasterio.features import rasterize
from shapely.geometry import box as shape_box
from rasterio.windows import Window
from collections import defaultdict
def burn_geom(geom,win,ds):
    if geom is None or geom.is_empty:return np.zeros((CHIP,CHIP),dtype=bool)
    window_box=shape_box(*rasterio.windows.bounds(win,ds.transform))
    if not geom.intersects(window_box):return np.zeros((CHIP,CHIP),dtype=bool)
    clipped=geom.intersection(window_box)
    if clipped.is_empty:return np.zeros((CHIP,CHIP),dtype=bool)
    return rasterize([(clipped,1)],out_shape=(CHIP,CHIP),
           transform=rasterio.windows.transform(win,ds.transform),
           fill=0,dtype='uint8').astype(bool)
def burn_geoms(geometries,win,ds):
    window_box=shape_box(*rasterio.windows.bounds(win,ds.transform))
    inter=[(z.intersection(window_box),1) for z in geometries if z is not None and z.intersects(window_box)]
    if not inter:return np.zeros((CHIP,CHIP),dtype=bool)
    return rasterize(inter,out_shape=(CHIP,CHIP),
        transform=rasterio.windows.transform(win,ds.transform),
        fill=0,dtype='uint8').astype(bool)
# Raster training only needs the active training-group footprints; held-out
# reference positive crowns CANNOT leak into target masks.
TRAIN_REFS={}
for f in [3]:
    train=units.merge(OWNERS[['unit_type','unit_id','roi_mosaic_coverage_fraction']],on=['unit_type','unit_id'],validate='one_to_one')
    train=train[train.roi_mosaic_coverage_fraction.ge(.95)]
    TRAIN_REFS[f]={}
    for site in SITES:
        cs=robust_union02(list(train[(train.site==site)&(train.unit_type=='circle')].geometry),
                          f'train_circle_f{f}_{site}')
        ps=robust_union02(list(train[(train.site==site)&(train.unit_type=='plot')].geometry),
                          f'train_plot_f{f}_{site}')
        TRAIN_REFS[f][site]=(cs,ps)
# Precompute fast spatial index on the optional partial crowns.
# Only explicit crown geometry is allowed to supervise the partial area.


## 02 — Cache and train full-data production model

In [ ]:
import time,gc,os
from rasterio.windows import Window
from rasterio.features import rasterize
import torch,torch.nn as nn,torch.nn.functional as F
OUT3=OUT/'production_training'
CACHE_DIR=OUT3/'chip_cache'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
CROP=256
MIN_LABEL=250
CHUNK=40
SEED_GPU=20261118
RUN_TRAIN=True
EPOCHS=12
STEPS_PER_EPOCH=120
BATCH=12
assert len(PROBLEMS)==2 and set(PROBLEMS.unit_id)=={'Mixt1','Mixt2'}
assert len(CANDIDATES)>0 and CANDIDATES.eligible.any()
# All sources pooled; identical physical windows sampled once.
CAND=CANDIDATES[CANDIDATES.eligible].sort_values(['fold','mosaic','row','col','source','source_id']).drop_duplicates(['fold','mosaic','row','col']).copy()
CAND['source_id']=CAND.source_id.astype(str)
assert not any((q.source,q.source_id) in UNAVAILABLE_KEYS for q in CAND.itertuples())
# Use the actual geometry/label mask functions already checked above.
KEY=hashlib.sha256(json.dumps(dict(
    nb01_hash=SPLIT_CONTRACT['csv_sha256'],nb03_key=TRAIN['training_key'],candidate_digest=hashlib.sha256(pd.util.hash_pandas_object(CAND[['mosaic','row','col','source','source_id']],index=False).values.tobytes()).hexdigest(),v='nb07_v5_full_data_unified_1',chip=CROP,
    crop_min_label=MIN_LABEL,n_candidates=len(CAND),
    roots=[str(q) for q in [PLOT_PATH,CROWN_PATH,EXH_PATH,CIRCLE_PATH,ROI_PATH]+
           [Path(v) for v in V31['shared_annotations'].values()]],
    source_mtimes=[Path(q).stat().st_mtime_ns for q in
                   [PLOT_PATH,CROWN_PATH,EXH_PATH,CIRCLE_PATH,ROI_PATH,FOLD_CSV]+
                   [Path(v) for v in V31['shared_annotations'].values()]],
    mosaic_metadata=[(str(d['path']),d['path'].stat().st_size,d['path'].stat().st_mtime_ns)
                     for _,d in sorted(mosaics.items())],
    plot_background_assumed_exhaustive=True),sort_keys=True).encode()).hexdigest()
from shapely.ops import transform as shapely_transform
from pyproj import Transformer
def unified_mask(ds,win,fold,site,rgb_valid):
    c_train,p_train=TRAIN_REFS[fold][site]
    valid=rgb_valid&burn_geom(roi_union,win,ds)
    cm=burn_geom(c_train,win,ds)
    pm=burn_geom(p_train,win,ds)
    cp=burn_geom(circ_positive,win,ds)&cm
    pp=burn_geom(plot_pos,win,ds)&pm
    pn=burn_geom(plot_nonpos,win,ds)&pm
    bbox=shape_box(*rasterio.windows.bounds(win,ds.transform))
    if ds.crs!=rasterio.crs.CRS.from_user_input(REF_CRS):
        bbox=shapely_transform(Transformer.from_crs(ds.crs,REF_CRS,always_xy=True).transform,bbox)
    assert not bbox.intersects(HELDOUT[fold][site]),'Held-out chip intersection'
    q=partial[site].iloc[partial[site].sindex.query(bbox,predicate='intersects')]
    tp=burn_geoms(list(q.loc[q.musanga].geometry),win,ds)
    tn=burn_geoms(list(q.loc[~q.musanga].geometry),win,ds)
    positive=cp|pp|tp
    negative=(cm&~cp)|(pm&~pp)|pn|tn
    conflict=positive&negative
    w=(positive|negative)&~conflict&valid
    y=positive&w
    return y.astype('uint8'),w.astype('uint8')
print('Unified cache key:',KEY[:16],'candidate windows:',len(CAND))


In [ ]:
# Build or resume only unified V5 cache chunks.
from pathlib import Path
CHIP_ROWS=[]
n_hits=0;n_written=0
for (fold,mosaic),df in CAND.groupby(['fold','mosaic'],sort=True):
    df=df.sort_values(['source','row','col','source_id']).reset_index(drop=True)
    site=mosaics[mosaic]['site']
    groups=[]
    for start in range(0,len(df),CHUNK):
        subset=df.iloc[start:start+CHUNK]
        identifier=f"f{fold}_{Path(mosaic).stem}_{start:06d}_{KEY[:16]}"
        npz=CACHE_DIR/f'{identifier}.npz'
        qa=CACHE_DIR/f'{identifier}.csv'
        if npz.exists() and qa.exists():
            try:
                old=pd.read_csv(qa)
                # Preserve old cache. Check key, dimensions and A/B masks,
                # without decompressing the large RGB member or entire archive.
                with np.load(npz,allow_pickle=False) as z:
                    assert str(z['key'].item())==KEY
                    assert z['rgb'].shape==(len(old),CROP,CROP,3)
                    assert z['y'].shape==z['w'].shape==(len(old),CROP,CROP)
                    assert np.all(z['y']<=z['w'])
                assert all(old['fold'].eq(int(fold)))
                assert all(old['mosaic'].eq(mosaic))
                assert len(old)==len(subset)
                assert old['row'].astype(int).to_list()==subset['row'].astype(int).to_list()
                assert old['col'].astype(int).to_list()==subset['col'].astype(int).to_list()
                CHIP_ROWS.extend(old.to_dict('records'))
                n_hits+=1
                print('REUSED',identifier,'rows',len(old),flush=True)
                continue
            except Exception as err:
                # Explicitly stop instead of deleting any existing work.
                raise RuntimeError(f'Existing chunk failed integrity check: {npz}. '
                                   'No original chunk was deleted. Inspect it manually.') from err
        # If interrupted between NPZ and CSV creation, save an untouched
        # backup of the orphan before rebuilding the pair.
        if npz.exists()!=qa.exists():
            from datetime import datetime,timezone
            orphan=npz if npz.exists() else qa
            stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
            backup=orphan.with_name(orphan.name+'.orphan_backup_'+stamp)
            orphan.rename(backup)
            print('PRESERVED incomplete orphan:',backup,'; rebuilding chunk',flush=True)
        groups.append((subset,identifier,npz,qa))
    if not groups:continue
    with rasterio.open(mosaics[mosaic]['path']) as ds:
        assert ds.count>=3
        for subset,identifier,npz,qa in groups:
            rgb_list=[];y_list=[];w_list=[];meta=[]
            for pos,rec in enumerate(subset.itertuples(index=False)):
                win=Window(int(rec.col),int(rec.row),CROP,CROP)
                data=ds.read([1,2,3],window=win,masked=True)
                assert data.shape==(3,CROP,CROP)
                rgb=np.asarray(data.filled(0))
                assert rgb.dtype==np.uint8,('Expected uint8 RGB',mosaic,rgb.dtype)
                valid=~np.any(np.ma.getmaskarray(data),axis=0)
                if ds.count>=4:
                    alpha=ds.read(4,window=win,masked=True)
                    valid &=~np.ma.getmaskarray(alpha)
                    valid &=np.asarray(alpha.filled(0))>0
                valid &=np.any(rgb!=0,axis=0)&np.any(rgb!=255,axis=0)
                valid &=~((rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255))
                y,w=unified_mask(ds,win,int(fold),site,valid)
                rgb_list.append(rgb.transpose(1,2,0).copy())
                y_list.append(y);w_list.append(w)
                meta.append(dict(fold=int(fold),site=site,mosaic=mosaic,
                    source=str(rec.source),source_id=str(rec.source_id),
                    row=int(rec.row),col=int(rec.col),cache_file=str(npz),
                    cache_index=pos,n_label=int(w.sum()),n_positive=int(y.sum())))
            arrays=dict(key=np.array(KEY),rgb=np.stack(rgb_list),y=np.stack(y_list),w=np.stack(w_list))
            tmp=npz.with_name(npz.name+'.tmp.npz')
            try:
                # Store lossless, uncompressed arrays to avoid repeated zlib CPU cost.
                # Old np.savez_compressed files remain fully supported by np.load.
                np.savez(tmp,**arrays)
                tmp.replace(npz)
                pd.DataFrame(meta).to_csv(qa,index=False)
            finally:
                tmp.unlink(missing_ok=True)
            CHIP_ROWS.extend(meta)
            n_written+=1
            print('CACHED',identifier,'rows',len(meta),'usable',sum(v['n_label']>=MIN_LABEL for v in meta),flush=True)
            del arrays,rgb_list,y_list,w_list
    gc.collect()
print('Cache chunks reused:',n_hits,'| newly built:',n_written)
CHIP_QA=pd.DataFrame(CHIP_ROWS)
assert len(CHIP_QA)==len(CAND),(len(CHIP_QA),len(CAND))
assert not CHIP_QA.duplicated(['fold','mosaic','source','row','col']).any()
assert CHIP_QA.n_positive.le(CHIP_QA.n_label).all()
CHIP_QA['usable']=CHIP_QA.n_label.ge(MIN_LABEL)
assert CHIP_QA.groupby(['fold','site']).usable.sum().gt(0).all()
CHIP_QA.to_csv(OUT3/'03_complete_chip_QA.csv',index=False)
display(CHIP_QA[CHIP_QA.usable].groupby(['fold','site','source']).agg(chips=('row','size'),positive=('n_positive','sum'),supervised=('n_label','sum')))
print('Unified cache complete:',len(CHIP_QA),'rows')


In [ ]:
# SECTION 06A — Load a single outer-training fold into RAM (not all three).
# ~1 GB/fold uncompressed, depending on available candidates. Avoid Drive random reads.
import segmentation_models_pytorch as smp
class ResNet18UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
                          in_channels=3,classes=1)
    def forward(self,x):return self.net(x)[:,0]

def load_fold(fold):
    frame=CHIP_QA[CHIP_QA.fold.eq(fold)].copy()
    pieces=[];rows=[]
    for p,part in frame.groupby('cache_file',sort=True):
        with np.load(p,allow_pickle=False) as z:
            assert str(z['key'].item())==KEY
            x=np.array(z['rgb']);y=np.array(z['y']);w=np.array(z['w'])
        assert len(x)==len(part)
        origin=len(rows)
        for rec in part.itertuples(index=False):
            q=rec._asdict()
            q['memory_index']=origin+int(rec.cache_index)
            rows.append(q)
        pieces.append((x,y,w))
    arrays={key:np.concatenate([v[k] for v in pieces],axis=0)
            for k,key in enumerate(['rgb','y','w'])}
    del pieces
    return pd.DataFrame(rows),arrays

def balanced_draw(frame,n,rng):
    by_site={site:frame.loc[frame.site.eq(site)] for site in SITES}
    available=[site for site in SITES if len(by_site[site])>0]
    assert available,'No usable sampled sites'
    chosen=[]
    for j in range(n):
        site=available[j%len(available)]
        sub=by_site[site]
        chosen.append(sub.iloc[int(rng.integers(len(sub)))])
    return chosen
def make_batch(recs,arrays,epoch,step):
    X=[];Y=[];W=[]
    for index,q in enumerate(recs):
        k=int(q.memory_index)
        x=arrays['rgb'][k];y=arrays['y'][k];w=arrays['w'][k]
        # Deterministic augmentation.
        h=hashlib.sha256(repr((k,epoch,step,index)).encode()).hexdigest()
        rng=np.random.default_rng(int(h[:15],16))
        rot=int(rng.integers(4));flip=bool(rng.integers(2))
        x=np.rot90(x,rot);y=np.rot90(y,rot);w=np.rot90(w,rot)
        if flip:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        X.append(np.ascontiguousarray(x.transpose(2,0,1)))
        Y.append(np.ascontiguousarray(y))
        W.append(np.ascontiguousarray(w))
    return (torch.from_numpy(np.stack(X).astype('float32')/255.),
            torch.from_numpy(np.stack(Y).astype('float32')),
            torch.from_numpy(np.stack(W).astype('float32')))
def masked_bce(logits,y,w):
    values=F.binary_cross_entropy_with_logits(logits.float(),y,reduction='none')
    return (values*w).sum()/w.sum().clamp_min(1)
if RUN_TRAIN:
    assert torch.cuda.is_available(),'Please select Runtime > Change runtime type > GPU (T4/L4)'
    print('GPU:',torch.cuda.get_device_name(0))
else:print('RUN_TRAIN=False: cache only')


In [ ]:
# SECTION 06B — Resumable A/B fits; each fold and model has its own checkpoint.
TRAIN_KEY=hashlib.sha256(json.dumps(dict(
    cache=KEY,seed=SEED_GPU,epochs=EPOCHS,steps=STEPS_PER_EPOCH,
    batch=BATCH,lr=2e-4,weight_decay=1e-4,
    architecture='smp_Unet_ResNet18',loss='masked_BCE'),sort_keys=True).encode()).hexdigest()
def fit_one(fold,frame,arrays):
    arm='unified'
    target=OUT3/f'fold{fold}_{arm}_{TRAIN_KEY[:18]}.pt'
    torch.manual_seed(SEED_GPU+fold)
    torch.cuda.manual_seed_all(SEED_GPU+fold)
    model=ResNet18UNet().to('cuda')
    optim=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    start=0;history=[]
    if target.exists():
        ck=torch.load(target,map_location='cpu',weights_only=False)
        assert ck['training_key']==TRAIN_KEY and ck['fold']==fold and ck['arm']==arm
        model.load_state_dict(ck['model'])
        optim.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        start=int(ck['epoch']);history=list(ck['history'])
        print('RESUME',arm,fold,'from epoch',start)
    pool=frame[frame.usable].copy()
    assert len(pool)>30 and set(pool.site)==set(SITES)
    for ep in range(start,EPOCHS):
        model.train();losses=[];begin=time.perf_counter()
        for step in range(STEPS_PER_EPOCH):
            rng=np.random.default_rng(SEED_GPU+fold*100000+ep*1000+step)
            recs=balanced_draw(pool,BATCH,rng)
            x,y,w=make_batch(recs,arrays,ep,step)
            x=x.cuda(non_blocking=True);y=y.cuda(non_blocking=True);w=w.cuda(non_blocking=True)
            optim.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                logits=model(x)
                loss=masked_bce(logits,y,w)
            scaler.scale(loss).backward()
            scaler.step(optim);scaler.update()
            losses.append(float(loss.detach().cpu()))
        history.append(dict(fold=fold,arm=arm,epoch=ep+1,
                            masked_bce=float(np.mean(losses)),
                            elapsed_seconds=time.perf_counter()-begin,
                            optimizer_steps=STEPS_PER_EPOCH,
                            training_examples=BATCH*STEPS_PER_EPOCH))
        checkpoint=dict(training_key=TRAIN_KEY,fold=fold,arm=arm,epoch=ep+1,
                        model=model.state_dict(),optimizer=optim.state_dict(),
                        scaler=scaler.state_dict(),history=history)
        tmp=target.with_name(target.name+'.tmp')
        torch.save(checkpoint,tmp);tmp.replace(target)
        pd.DataFrame(history).to_csv(OUT3/f'fold{fold}_{arm}_history.csv',index=False)
        print('TRAIN',arm,'fold',fold,'epoch',ep+1,'/',EPOCHS,
              'BCE',round(history[-1]['masked_bce'],4),
              's',round(history[-1]['elapsed_seconds'],1),flush=True)
    del model,optim,scaler
    torch.cuda.empty_cache()
    gc.collect()
if RUN_TRAIN:
    for fold in [3]:
        print('Loading cached RGB for fold',fold,flush=True)
        rows,arrays=load_fold(fold)
        fit_one(fold,rows,arrays)
        del rows,arrays;gc.collect()
    print('FULL-DATA PRODUCTION MODEL TRAINED. Not an independent validation model.')


## 03 — Fingerprinted grouped inference, resumable by stripe

In [ ]:
import time, shutil
from rasterio.windows import from_bounds,transform as win_transform
from rasterio.enums import Resampling
from shapely.prepared import prep
from rasterio.errors import NodataShadowWarning
import warnings
warnings.filterwarnings('ignore',category=NodataShadowWarning)
CORE=256;HALO=64;GROUP=4;STRIPE=1024;BATCH_INFER=8
ROI_SETBACK_M=100.;THRESHOLD=.5
SITE='ALL' # all six mosaics by default
ROI_INTERIOR=roi_union.buffer(-ROI_SETBACK_M)
assert ROI_INTERIOR.is_valid and not ROI_INTERIOR.is_empty
CHECKPOINT=OUT3/f'fold3_unified_{TRAIN_KEY[:18]}.pt'
ck=torch.load(CHECKPOINT,map_location='cpu',weights_only=False)
assert ck['training_key']==TRAIN_KEY and ck['epoch']==EPOCHS and ck['fold']==3 and ck['arm']=='unified'
net=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
net.load_state_dict({k.removeprefix('net.'):v for k,v in ck['model'].items()},strict=True)
net=net.cuda().eval()
del ck
def file_sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
MODEL_SHA=file_sha(CHECKPOINT)
POLICY=dict(schema='nb07_rgb_v5_1',core=CORE,halo=HALO,stripe=STRIPE,group=GROUP,threshold=THRESHOLD,roi_setback_m=ROI_SETBACK_M,roi_sha=hashlib.sha256(ROI_INTERIOR.wkb).hexdigest(),model_sha=MODEL_SHA,training_key=TRAIN_KEY,precision='float32',validity='RGB mask and alpha; exclude black white blue; ROI interior')
POLICY_SHA=hashlib.sha256(json.dumps(POLICY,sort_keys=True).encode()).hexdigest()
MERGED=OUT/'reconstructed_mosaics';MERGED.mkdir(exist_ok=True)
SCRATCH=Path('/content/musanga_v5_nb07');SCRATCH.mkdir(exist_ok=True)
def atomic_json(p,obj):
 tmp=p.with_suffix(p.suffix+'.tmp');tmp.write_text(json.dumps(obj,indent=2));os.replace(tmp,p)
atomic_json(OUT/'07_production_model_contract.json',{**POLICY,'policy_sha':POLICY_SHA,'checkpoint':str(CHECKPOINT),'n_training_chips':int(CHIP_QA.usable.sum()),'full_data':True,'independent_validation':False})
def profile(w,h,crs,tr):
 return dict(driver='GTiff',width=int(w),height=int(h),count=2,dtype='float32',nodata=-9999,crs=crs,transform=tr,tiled=True,blockxsize=256,blockysize=256,compress='DEFLATE',predictor=3,BIGTIFF='YES',SPARSE_OK='TRUE')
def mark(ds,key):
 ds.set_band_description(1,'Musanga probability')
 ds.set_band_description(2,'Musanga binary at 0.5')
 ds.update_tags(signature=key,model_sha=MODEL_SHA,policy_sha=POLICY_SHA)
def checked(p,key,w,h,tr,crs):
 if not p.is_file():return False
 try:
  with rasterio.open(p) as ds:
   return ds.width==w and ds.height==h and ds.count==2 and ds.dtypes==('float32','float32') and ds.nodata==-9999 and ds.crs==crs and ds.transform.almost_equals(tr) and ds.tags().get('signature')==key
 except Exception:return False
def read_group(ds,row,col,h,w):
 # Read contiguous native RGB and alpha with explicit exterior padding.
 r=row-HALO;c=col-HALO;hh=h+2*HALO;ww=w+2*HALO
 bands=[1,2,3]+([4] if ds.count>=4 else [])
 out=np.zeros((3,hh,ww),np.uint8);valid=np.zeros((hh,ww),bool)
 rr=max(0,r);cc=max(0,c);r1=min(ds.height,r+hh);c1=min(ds.width,c+ww)
 if r1<=rr or c1<=cc:return out,valid
 a=ds.read(bands,window=Window(cc,rr,c1-cc,r1-rr),masked=True)
 rgb=a[:3].filled(0);v=~np.ma.getmaskarray(a[:3]).any(axis=0)
 if len(bands)==4:v &= (~np.ma.getmaskarray(a[3]))&(a[3].filled(0)>0)
 v &= np.any(rgb!=0,axis=0)&np.any(rgb!=255,axis=0)
 v &= ~((rgb[0]==0)&(rgb[1]==0)&(rgb[2]==255))
 ys=slice(rr-r,r1-r);xs=slice(cc-c,c1-c)
 out[:,ys,xs]=rgb;valid[ys,xs]=v
 return out,valid
@torch.inference_mode()
def infer_group(ds,geom,row,col,h,w):
 rgb,valid=read_group(ds,row,col,h,w)
 roi=rasterize([(geom,1)],out_shape=(h,w),transform=win_transform(Window(col,row,w,h),ds.transform),fill=0,dtype='uint8').astype(bool)
 support=roi&valid[HALO:HALO+h,HALO:HALO+w]
 result=np.full((2,h,w),-9999.,np.float32)
 if not support.any():return result,0,0
 tasks=[]
 for r in range(0,h,CORE):
  for c in range(0,w,CORE):
   hh=min(CORE,h-r);ww=min(CORE,w-c)
   if not support[r:r+hh,c:c+ww].any():continue
   x=np.zeros((3,CORE+2*HALO,CORE+2*HALO),np.uint8)
   a=rgb[:,r:r+hh+2*HALO,c:c+ww+2*HALO]
   x[:,:a.shape[1],:a.shape[2]]=a
   tasks.append((r,c,hh,ww,x))
 for start in range(0,len(tasks),BATCH_INFER):
  batch=tasks[start:start+BATCH_INFER]
  x=torch.from_numpy(np.stack([t[4] for t in batch])).cuda().float()/255.
  p=torch.sigmoid(net(x)[:,0]).cpu().numpy()
  for (r,c,hh,ww,_),v in zip(batch,p):
   core=v[HALO:HALO+hh,HALO:HALO+ww];ok=support[r:r+hh,c:c+ww]
   assert np.isfinite(core[ok]).all()
   result[0,r:r+hh,c:c+ww][ok]=core[ok]
   result[1,r:r+hh,c:c+ww][ok]=(core[ok]>=THRESHOLD)
 return result,int(support.sum()),len(tasks)
def run_mosaic(name,d):
 p=d['path'];st=p.stat()
 key=hashlib.sha256(repr((POLICY_SHA,str(p),st.st_size,st.st_mtime_ns)).encode()).hexdigest()
 stem=Path(name).stem
 stripes=OUT/'stripe_cache'/stem/key[:20];stripes.mkdir(parents=True,exist_ok=True)
 final=MERGED/f'{stem}_Musanga_probability_binary_v5.tif'
 report=MERGED/f'{stem}_reconstruction.json'
 with rasterio.open(p) as src:
  assert src.crs==rasterio.crs.CRS.from_user_input(REF_CRS) and src.count>=3 and src.dtypes[:3]==('uint8',)*3
  geom=ROI_INTERIOR.intersection(box(*src.bounds))
  assert not geom.is_empty,(name,'empty interior')
  # Crop to the native core-aligned ROI bounding window.
  raw=from_bounds(*geom.bounds,transform=src.transform)
  c0=max(0,int(np.floor(raw.col_off/CORE))*CORE);r0=max(0,int(np.floor(raw.row_off/CORE))*CORE)
  c1=min(src.width,int(np.ceil((raw.col_off+raw.width)/CORE))*CORE);r1=min(src.height,int(np.ceil((raw.row_off+raw.height)/CORE))*CORE)
  w=c1-c0;h=r1-r0;tr=win_transform(Window(c0,r0,w,h),src.transform)
  assert w>0 and h>0
  if final.exists():
   assert report.exists() and checked(final,key,w,h,tr,src.crs)
   rec=json.loads(report.read_text());assert rec['status']=='complete' and rec['digest']==key
   print('COMPLETE CACHE',name,flush=True);return rec
  prepared=prep(geom);nstripes=math.ceil(h/STRIPE);records=[];t0=time.monotonic()
  for i in range(nstripes):
   y=i*STRIPE;hh=min(STRIPE,h-y);stf=win_transform(Window(c0,r0+y,w,hh),src.transform)
   dst=stripes/f'{i:05d}.tif';meta=dst.with_suffix('.json')
   if checked(dst,key,w,hh,stf,src.crs) and meta.exists():
    rec=json.loads(meta.read_text());assert rec['signature']==key and rec['stripe']==i
   else:
    local=SCRATCH/f'{stem}_{i:05d}.tif'
    valid_count=0;tile_count=0
    with rasterio.open(local,'w',**profile(w,hh,src.crs,stf)) as out:
     mark(out,key)
     for rr in range(0,hh,CORE*GROUP):
      bh=min(CORE*GROUP,hh-rr)
      for cc in range(0,w,CORE*GROUP):
       bw=min(CORE*GROUP,w-cc)
       sw=Window(c0+cc,r0+y+rr,bw,bh)
       if not prepared.intersects(box(*rasterio.windows.bounds(sw,src.transform))):continue
       block,nv,nt=infer_group(src,geom,r0+y+rr,c0+cc,bh,bw)
       if nv:out.write(block,window=Window(cc,rr,bw,bh))
       valid_count+=nv;tile_count+=nt
    pending=dst.with_suffix('.uploading')
    shutil.copyfile(local,pending);os.replace(pending,dst);local.unlink()
    assert checked(dst,key,w,hh,stf,src.crs)
    rec=dict(signature=key,stripe=i,valid_pixels=valid_count,tiles=tile_count)
    atomic_json(meta,rec)
   records.append(rec)
   print(f'{name} stripe {i+1}/{nstripes} | valid {rec["valid_pixels"]:,} | tiles {rec["tiles"]:,} | {(time.monotonic()-t0)/60:.1f} min',flush=True)
  assert sum(z['valid_pixels'] for z in records)>0
  # Assemble directly to one sparse BigTIFF on Drive; checkpoint per stripe.
  partial=MERGED/f'{stem}_{key[:12]}.partial.tif'
  progress=partial.with_suffix('.json')
  if not partial.exists():
   with rasterio.open(partial,'w',**profile(w,h,src.crs,tr)) as out:mark(out,key)
   atomic_json(progress,{'signature':key,'next_stripe':0})
  assert checked(partial,key,w,h,tr,src.crs)
  state=json.loads(progress.read_text());assert state['signature']==key
  for i in range(int(state['next_stripe']),nstripes):
   with rasterio.open(stripes/f'{i:05d}.tif') as a,rasterio.open(partial,'r+') as out:
    for _,win in a.block_windows(1):
     z=a.read(window=win);valid=z[0]!=-9999
     assert np.array_equal(valid,z[1]!=-9999)
     assert np.all((z[0][valid]>=0)&(z[0][valid]<=1))
     assert np.array_equal(z[1][valid],(z[0][valid]>=THRESHOLD).astype(np.float32))
     if valid.any():out.write(z,window=Window(win.col_off,win.row_off+i*STRIPE,win.width,win.height))
   atomic_json(progress,{'signature':key,'next_stripe':i+1})
   print(f'{name} assembly {i+1}/{nstripes}',flush=True)
  assert checked(partial,key,w,h,tr,src.crs)
  rec=dict(name=stem,site=d['site'],status='complete',output=str(final),width=w,height=h,valid_pixels=sum(z['valid_pixels'] for z in records),count=sum(z['tiles'] for z in records),model=MODEL_SHA,policy=POLICY_SHA,digest=key,roi_setback_m=ROI_SETBACK_M,threshold=THRESHOLD,transform=list(tr),crs=str(src.crs))
  atomic_json(report,rec);os.replace(partial,final)
  print('COMPLETE',name,flush=True)
  return rec
print('Production model:',MODEL_SHA[:16],'| policy:',POLICY_SHA[:16])


## 04 — Run all six mosaics and save manifest

In [ ]:
selected={k:v for k,v in mosaics.items() if SITE=='ALL' or v['site']==SITE}
assert selected
reports=[run_mosaic(k,v) for k,v in selected.items()]
frame=pd.DataFrame(reports)
display(frame[['name','site','width','height','valid_pixels','status']])
if SITE=='ALL':
 assert len(frame)==6 and frame.status.eq('complete').all()
 frame.to_csv(OUT/'07_production_mosaic_manifest.csv',index=False)
 atomic_json(OUT/'07_production_mosaic_manifest.json',{'schema':'musanga_v5_nb07_rgb_mosaics','model_sha':MODEL_SHA,'policy_sha':POLICY_SHA,'mosaics':reports,'complete':True})
 print('All six native RGB production mosaics complete:',MERGED)
else:print('Subset only; run SITE=ALL to create complete NB08 handoff.')


## 05 — Interpretation and handoff

Full-data products are model-derived targets, not independent ground truth. NB04 remains the independent spatial holdout assessment. Native probability and binary maps share valid support; all invalid pixels are NoData. NB08 must use the manifest and dynamic signatures rather than historical fixed paths or fingerprints. Stripe caches remain available for recovery until final V5 clean-run verification.